# HardSigmoid

参考：[HardSigmoid](https://onnx.ai/onnx/operators/onnx__HardSigmoid.html)

HardSigmoid 函数

$$
\operatorname{HardSigmoid}(x) = \max(0, \min(1, \alpha x + \beta))，
$$

通常情况 $\alpha=0.2$ 和 $\beta=0.5$。

In [1]:
from pathlib import Path
temp_dir = Path(".temp")
temp_dir.mkdir(exist_ok=True)
model_path = f"{temp_dir}/HardSigmoid.onnx" # 模型存储路径

构建模型：

In [2]:
from onnxscript import opset20 as op
from onnxscript import ir
from onnxscript import script
from onnxscript import FLOAT
import onnx

@script()
def model(x: FLOAT[1, 3, 4, 4]) -> FLOAT[1, 3, 4, 4]:
    x = op.Add(x, x)
    y = op.HardSigmoid(x)
    x = op.Mul(x, y)
    x = op.HardSigmoid(x)
    return x
    # return op.HardSwish(x,)

onnx.save_model(model.to_model_proto(), model_path,)

/media/pc/data/lxw/envs/anaconda3a/envs/ai/lib/python3.12/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()


## 转换为 Relay 模型

In [3]:
from tvm.driver.tvmc.frontends import load_model

model = load_model(model_path)
model.mod.show()

In [4]:
import numpy as np
from tvm import relay
import tvm
from tvm.relay.dataflow_pattern import (
    wildcard, is_op, 
    is_constant, 
    # is_tuple,
    # FunctionPattern,
    DFPatternCallback,
    rewrite
)
from tvm.relay import op as _op
from tvm.relay import transform as _transform
from tvm.relay.frontend.common import infer_value

@tvm.register_func
def hard_sigmoid(x):
    print(x)
    print(type(x))
    return _op.clip(_op.multiply(x, _op.const(1/6)) + _op.const(0.5), 0.0, 1.0)

class HardSigmoidV1Rewrite(DFPatternCallback):
    def __init__(self):
        super().__init__()
        self.x = wildcard()
        self.cons1 = is_constant()
        self.add = is_op("add")(self.x, self.cons1)
        self.clip = is_op("clip")(self.add)
        self.cons2 = is_constant()
        self.divide = is_op("divide")(self.clip, self.cons2)
        self.pattern = self.divide

    def callback(self, pre, post, node_map):
        x = node_map[self.x][0]
        clip = node_map[self.clip][0]
        _transform.InferTypeLocal(x)
        cons1 = node_map[self.cons1][0]
        cons2 = node_map[self.cons2][0]
        if np.allclose(cons1.data.numpy(), 3.0) and np.allclose(cons2.data.numpy(), 6.0) and np.allclose(clip.attrs.a_min, 0.0) and np.allclose(clip.attrs.a_max, 6.0):
            out = hard_sigmoid(x)
            _transform.InferTypeLocal(out)
        else:
            out = post
        return out

class HardSigmoidV2Rewrite(DFPatternCallback):
    def __init__(self):
        super().__init__()
        self.x = wildcard()
        self.cons1 = is_constant()
        self.multiply = is_op("multiply")(self.x, self.cons1)
        self.cons2 = is_constant()
        self.add = is_op("add")(self.multiply, self.cons2)
        self.clip = is_op("clip")(self.add)
        self.pattern = self.clip

    def callback(self, pre, post, node_map):
        x = node_map[self.x][0]
        clip = node_map[self.clip][0]
        _transform.InferTypeLocal(x)
        cons1 = node_map[self.cons1][0]
        cons2 = node_map[self.cons2][0]
        if np.allclose(cons1.data.numpy(), 0.2) and np.allclose(cons2.data.numpy(), 0.5) and np.allclose(clip.attrs.a_min, 0.0) and np.allclose(clip.attrs.a_max, 1.0):
            out = tvm.get_global_func("hard_sigmoid")(x)
            # out = _op.clip(x, 0, 1)
            _transform.InferTypeLocal(out)
        else:
            out = post
        return out

@tvm.transform.module_pass(opt_level=1)
class FuseHardSigmoid:
    """融合 HardSigmoid"""
    def transform_module(self, mod, ctx):
        mod["main"] = rewrite(HardSigmoidV1Rewrite(), mod["main"])
        mod["main"] = rewrite(HardSigmoidV2Rewrite(), mod["main"])
        return mod


In [5]:
mod = FuseHardSigmoid()(model.mod)
mod.show()

free_var %x: Tensor[(1, 3, 4, 4), float32] /* ty=Tensor[(1, 3, 4, 4), float32] span=n0.x:0:0 */;
add(%x, %x) /* ty=Tensor[(1, 3, 4, 4), float32] span=n0:0:0 */
<class 'tvm.relay.expr.Call'>
free_var %x: Tensor[(1, 3, 4, 4), float32] /* ty=Tensor[(1, 3, 4, 4), float32] span=n0.x:0:0 */;
%0 = add(%x, %x) /* ty=Tensor[(1, 3, 4, 4), float32] span=n0:0:0 */;
%1 = multiply(%0, 0.166667f);
%2 = add(%1, 0.5f);
%3 = clip(%2, a_min=0f, a_max=1f) /* ty=Tensor[(1, 3, 4, 4), float32] */;
multiply(%0, %3) /* ty=Tensor[(1, 3, 4, 4), float32] span=n2:0:0 */
<class 'tvm.relay.expr.Call'>


In [9]:
simplify = relay.transform.SimplifyInference()
mod = relay.transform.InferType()(mod)
mod = simplify(mod)
print(mod)

def @main(%x: Tensor[(1, 3, 4, 4), float32] /* ty=Tensor[(1, 3, 4, 4), float32] span=n0.x:0:0 */) -> Tensor[(1, 3, 4, 4), float32] {
  %0 = add(%x, %x) /* ty=Tensor[(1, 3, 4, 4), float32] span=n0:0:0 */;
  %1 = multiply(%0, 0.166667f /* ty=float32 */) /* ty=Tensor[(1, 3, 4, 4), float32] */;
  %2 = add(%1, 0.5f /* ty=float32 */) /* ty=Tensor[(1, 3, 4, 4), float32] */;
  %3 = clip(%2, a_min=0f, a_max=1f) /* ty=Tensor[(1, 3, 4, 4), float32] */;
  %4 = multiply(%0, %3) /* ty=Tensor[(1, 3, 4, 4), float32] span=n2:0:0 */;
  %5 = multiply(%4, 0.166667f /* ty=float32 */) /* ty=Tensor[(1, 3, 4, 4), float32] */;
  %6 = add(%5, 0.5f /* ty=float32 */) /* ty=Tensor[(1, 3, 4, 4), float32] */;
  clip(%6, a_min=0f, a_max=1f) /* ty=Tensor[(1, 3, 4, 4), float32] */
}



In [6]:
qq

NameError: name 'qq' is not defined

In [ ]:

from tvm.contrib.msc.core.frontend import translate
from tvm.contrib.msc.framework.tvm import codegen as tvm_codegen


opt_config = {"opt_level": 3}
graph, weights = translate.from_relay(model.mod, model.params, opt_config=opt_config)
codegen_config = {"explicit_name": False, "from_relay": True}
rt_mod = tvm_codegen.to_relax(graph, weights, codegen_config)
rt_mod.show()

In [ ]:
relay.clip